# 04 · Analyse SQL du Réseau Électrique
## CEET Smart Grid – Energy Blackout Prediction
**Objectif :** Créer une base SQLite et effectuer des requêtes analytiques avancées.

In [1]:
import sys
sys.path.insert(0, '../src')
import pandas as pd
import sqlite3
from sql_queries import create_database, run_query, run_all_queries, custom_query, QUERIES
from utils import DATA_PROC

df = pd.read_csv(DATA_PROC / 'ceet_processed.csv')
print(f"Dataset chargé : {df.shape}")

Dataset chargé : (50000, 75)


### 4.1 Création de la Base SQLite

In [2]:
conn = create_database(df)
print("Base SQLite créée avec succès")

# Vérification des tables
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' OR type='view'", conn)
print("Tables & Vues :")
print(tables)

2026-06-10 14:20:59 | INFO     | sql | Connexion SQLite : D:\Data_Science_Lab\Data_Project_Capstone_IBM\energy-grid-blackout-prediction\data\processed\ceet_smartgrid.db
2026-06-10 14:21:02 | INFO     | sql | Table 'grid_readings' : 50,000 lignes
2026-06-10 14:21:04 | INFO     | sql | Vues créées : v_blackout_events, v_overload_events


Base SQLite créée avec succès
Tables & Vues :
                name
0      grid_readings
1  v_blackout_events
2  v_overload_events


### 4.2 Vue d'Ensemble du Réseau

In [3]:
overview = run_query('overview', conn)
print("=== APERÇU GLOBAL ===")
for col in overview.columns:
    print(f"  {col:30s}: {overview[col].iloc[0]}")

2026-06-10 14:21:06 | INFO     | sql | Requête 'overview' → 1 lignes


=== APERÇU GLOBAL ===
  total_readings                : 50000
  total_blackouts               : 1975
  total_overloads               : 41234
  avg_load_mw                   : 310.33
  avg_outage_risk               : 94.61
  avg_shedding_mw               : 14.37
  period_start                  : 2020-01-01 00:00:00
  period_end                    : 2025-09-14 07:00:00


### 4.3 Taux de Blackout par Région

In [4]:
df_region = run_query('blackout_by_region', conn)
print(df_region.to_string(index=False))

2026-06-10 14:21:39 | INFO     | sql | Requête 'blackout_by_region' → 6 lignes


  region  total  blackouts  blackout_rate_pct  avg_risk  avg_shedding
 Savanes   8351        360               4.31     94.56         14.30
Maritime   8270        337               4.07     94.67         14.44
    Lome   8237        320               3.88     94.70         14.30
Plateaux   8509        328               3.85     94.58         14.52
    Kara   8236        314               3.81     94.61         14.48
Centrale   8397        316               3.76     94.53         14.20


### 4.4 Analyse Mensuelle

In [5]:
monthly = run_query('monthly_summary', conn)
print("Données mensuelles :")
print(monthly[['month','blackouts','overloads','avg_load_mw','avg_risk','total_shedding_mw']].to_string(index=False))

2026-06-10 14:21:44 | INFO     | sql | Requête 'monthly_summary' → 69 lignes


Données mensuelles :
  month  blackouts  overloads  avg_load_mw  avg_risk  total_shedding_mw
2020-01         34        628       311.39     94.98           11019.99
2020-02         24        567       307.15     94.14            9846.35
2020-03         35        598       309.49     94.46           10283.86
2020-04         28        599       311.51     94.67           10405.55
2020-05         23        593       308.36     94.19           10260.65
2020-06         31        582       309.42     94.54            9977.76
2020-07         39        620       309.38     95.02           10726.61
2020-08         25        587       307.48     94.13           10021.03
2020-09         27        595       309.97     94.86           10506.07
2020-10         40        623       314.27     94.98           10416.31
2020-11         20        590       307.81     94.31           10272.61
2020-12         25        599       307.44     94.25           10454.05
2021-01         20        624       310.36 

### 4.5 Périodes de Risque Maximum

In [6]:
top_risk = run_query('top_risk_periods', conn)
print(f"Top {len(top_risk)} périodes à risque critique (>90%):")
print(top_risk[['datetime','region','city','outage_risk','total_load_mw','event']].head(15).to_string(index=False))

2026-06-10 14:21:49 | INFO     | sql | Requête 'top_risk_periods' → 50 lignes


Top 50 périodes à risque critique (>90%):
           datetime   region     city  outage_risk  total_load_mw             event
2020-01-01 00:00:00     Lome   Sokode        100.0         352.87 Match de Football
2020-01-01 02:00:00 Centrale  Kpalime        100.0         298.21     Pas evenement
2020-01-01 04:00:00     Lome   Tsevie        100.0         355.56   Event Politique
2020-01-01 06:00:00 Maritime  Kpalime        100.0         361.83         Festivale
2020-01-01 09:00:00 Centrale     Lome        100.0         320.75           Concert
2020-01-01 11:00:00     Lome     Kara        100.0         323.30         Festivale
2020-01-01 12:00:00 Plateaux Atakpame        100.0         317.78 Match de Football
2020-01-01 13:00:00 Centrale     Lome        100.0         370.69           Concert
2020-01-01 14:00:00 Centrale  Kpalime        100.0         326.29         Festivale
2020-01-01 17:00:00 Maritime     Kara        100.0         315.13   Event Politique
2020-01-01 18:00:00 Maritime  Kpal

### 4.6 Fenêtre SQL — Charge par Heure

In [7]:
hourly_window = run_query('hourly_load_window', conn)
print("Analyse fenêtre temporelle (charge lissée) :")
print(hourly_window.to_string(index=False))

2026-06-10 14:21:53 | INFO     | sql | Requête 'hourly_load_window' → 24 lignes


Analyse fenêtre temporelle (charge lissée) :
 hour  avg_load  smoothed_load  max_load  min_load  blackouts
    0    298.75         135.12    456.38    145.92         78
    1    298.83         137.85    480.39    124.31         81
    2    297.95         139.78    474.06    135.14         78
    3    299.19         137.99    455.14    146.01         66
    4    299.03         144.36    454.67    147.50         94
    5    299.73         149.92    433.27    137.01         78
    6    299.90         144.25    439.31    156.15         87
    7    298.60         145.71    445.57    162.95         84
    8    298.68         144.33    444.78    117.63         78
    9    298.98         141.28    449.89    154.79         69
   10    298.28         140.61    437.90    130.14         78
   11    299.32         144.61    475.05    140.87         91
   12    299.48         140.14    439.72    159.60         69
   13    298.14         145.70    447.79    137.67         78
   14    298.29         1

### 4.7 Requête Personnalisée — Analyse des Transformateurs

In [8]:
sql_custom = '''
    SELECT
        region,
        city,
        ROUND(AVG(transformer_temp), 2)  AS avg_temp,
        MAX(transformer_temp)            AS max_temp,
        ROUND(AVG(voltage), 2)           AS avg_voltage,
        SUM(blackout)                    AS blackouts,
        SUM(overload)                    AS overloads,
        COUNT(*)                         AS observations,
        ROUND(100.0 * SUM(blackout) / COUNT(*), 2) AS blackout_rate_pct
    FROM grid_readings
    WHERE transformer_temp IS NOT NULL
    GROUP BY region, city
    HAVING COUNT(*) > 100
    ORDER BY avg_temp DESC
    LIMIT 15
'''
transformer_analysis = custom_query(sql_custom, conn)
print("=== Analyse des Transformateurs ===")
print(transformer_analysis.to_string(index=False))
conn.close()

=== Analyse des Transformateurs ===
  region     city  avg_temp  max_temp  avg_voltage  blackouts  overloads  observations  blackout_rate_pct
    Lome  Kpalime     70.59     97.56       219.96         46        929          1117               4.12
 Savanes Atakpame     70.53    102.65       220.15         56        987          1184               4.73
Maritime     Kara     70.49    108.98       220.05         61        962          1177               5.18
    Kara     Kara     70.40     98.31       219.95         59        956          1150               5.13
Maritime   Sokode     70.39     98.77       219.52         52        935          1154               4.51
 Savanes  Dapaong     70.37    100.48       219.83         46        985          1209               3.80
Maritime     Lome     70.36    104.18       219.57         52        933          1126               4.62
Centrale     Kara     70.30     99.14       220.36         50        988          1180               4.24
Centrale A